In [1]:
from neo4j import GraphDatabase
import os
import json
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()  # will read .env in current dir

True

In [2]:
uri = os.getenv("NEO4J_URI")
user = os.getenv("NEO4J_USERNAME")
password = os.getenv("NEO4J_PASSWORD")
DB  = os.getenv("NEO4J_DATABASE", "neo4j")

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"] 

driver = GraphDatabase.driver(uri, auth=(user, password))
print("[INFO] Connecting to:", uri)
print("[INFO] DB:", DB)
print("[INFO] USER:", user, "PWD:", password)
llm = ChatOpenAI(temperature=0, model_name="gpt-4o", api_key=OPENAI_API_KEY)

[INFO] Connecting to: neo4j+s://62b9e173.databases.neo4j.io
[INFO] DB: neo4j
[INFO] USER: neo4j PWD: oHiXwOTSqk3Tv9TkX-wrcngsjffZeIWIYkcvwwzBefQ


In [3]:
# Function to load JSON data
def load_knowledge_graph(json_file_path):
    with open(json_file_path, 'r') as file:
        return json.load(file)

In [10]:
def create_node(tx, node):
    # Initialize empty attributes dict if not present
    attributes = node.get("attributes", {})
    
    # ✅ Fallback: create 'name' from id (for meaningful embedding) if missing
    if "doc_id" in node and node["doc_id"] is not None:
        attributes["doc_id"] = node["doc_id"]
   
    label = f"`{node['label'].replace(' ', '_').replace('-', '_')}`"
    attributes_str = ", ".join([f"`{key}`: ${key}" for key in attributes.keys()])

    query = f"""
    MERGE (n:{label} {{id: $id}})
    {"SET n += {" + attributes_str + "}" if attributes_str else ""}
    """
    tx.run(query, id=node["id"], **attributes)


In [11]:
def create_relationship(tx, relationship):
    if "type" not in relationship or "source" not in relationship or "target" not in relationship:
        print(f"Skipping relationship due to missing fields: {relationship}")
        return

    attributes = relationship.get("attributes", {})  # Likely empty in your new format
    attributes_str = ", ".join([f"{key}: ${key}" for key in attributes.keys()])
    rel_type = f"`{relationship['type'].replace(' ', '_').replace('-', '_')}`"

    query = f"""
    MATCH (a {{id: $source}}), (b {{id: $target}})
    MERGE (a)-[r:{rel_type}]->(b)
    {"SET r += {" + attributes_str + "}" if attributes_str else ""}
    """
    tx.run(query, source=relationship["source"], target=relationship["target"], **attributes)


In [12]:
def store_knowledge_graph(driver, graph):
    START_FROM_NODE_INDEX = 0
    with driver.session() as session:
        print("[INFO] Storing Nodes...")
        total_nodes = len(graph["nodes"])
        for idx, node in enumerate(graph["nodes"][START_FROM_NODE_INDEX:], start=START_FROM_NODE_INDEX + 1):
            if "id" not in node or "label" not in node:
                print(f"[WARNING] Skipping node with missing 'id' or 'label': {node}")
                continue
            print(f"[INFO] Adding Node {idx}/{total_nodes}: ID = {node.get('id')}, Label = {node.get('label')}")
            session.write_transaction(create_node, node)

        print("[INFO] Storing Relationships...")
        for idx, relationship in enumerate(graph["relationships"], start=1):
            print(f"[INFO] Adding Relationship {idx}/{len(graph['relationships'])}: Type = {relationship.get('type')}, Source = {relationship.get('source')}, Target = {relationship.get('target')}")
            session.write_transaction(create_relationship, relationship)



In [13]:
# Load the knowledge graph data from a JSON file
json_file_path = "/mnt/SAS_A/srushti_thesis/Final_Code/PoisonedRAG/experiment/0_.json"   # Path to your JSON file
with open(json_file_path, "r") as file:
    knowledge_graph = json.load(file)

# Store the knowledge graph in Neo4j
try:
    store_knowledge_graph(driver, knowledge_graph)
    print("Knowledge graph stored in Neo4j successfully!")
finally:
    driver.close()

/tmp/ipykernel_2821539/127152020.py:3: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  with driver.session() as session:
/tmp/ipykernel_2821539/127152020.py:11: DeprecationWarning: write_transaction has been renamed to execute_write
  session.write_transaction(create_node, node)


[INFO] Storing Nodes...
[INFO] Adding Node 1/394: ID = accounting, Label = keyword
[INFO] Adding Node 2/394: ID = minority_interest, Label = scientific_term
[INFO] Adding Node 3/394: ID = non_controlling_interest, Label = scientific_term
[INFO] Adding Node 4/394: ID = subsidiary_corporation, Label = organization
[INFO] Adding Node 5/394: ID = parent_corporation, Label = organization
[INFO] Adding Node 6/394: ID = minority_interest_portion, Label = other
[INFO] Adding Node 7/394: ID = val_50, Label = value_number
[INFO] Adding Node 8/394: ID = special_voting_rights, Label = other
[INFO] Adding Node 9/394: ID = controlling_interest, Label = other
[INFO] Adding Node 10/394: ID = ownership, Label = other
[INFO] Adding Node 11/394: ID = accounting_standards, Label = other
[INFO] Adding Node 12/394: ID = minority_interest, Label = other
[INFO] Adding Node 13/394: ID = investors, Label = other
[INFO] Adding Node 14/394: ID = consolidated_balance_sheet, Label = other
[INFO] Adding Node 15/394:

/tmp/ipykernel_2821539/127152020.py:16: DeprecationWarning: write_transaction has been renamed to execute_write
  session.write_transaction(create_relationship, relationship)


[INFO] Adding Relationship 4/342: Type = LESS_THAN, Source = minority_interest_portion, Target = val_50
[INFO] Adding Relationship 5/342: Type = ACHIEVED_WITHOUT, Source = controlling_interest, Target = ownership
[INFO] Adding Relationship 6/342: Type = ACHIEVED_THROUGH, Source = controlling_interest, Target = special_voting_rights
[INFO] Adding Relationship 7/342: Type = DEPENDS_ON, Source = controlling_interest, Target = accounting_standards
[INFO] Adding Relationship 8/342: Type = BELONGS_TO, Source = minority_interest, Target = investors
[INFO] Adding Relationship 9/342: Type = REPORTED_ON, Source = minority_interest, Target = consolidated_balance_sheet
[INFO] Adding Relationship 10/342: Type = HAS, Source = owning_company, Target = consolidated_balance_sheet
[INFO] Adding Relationship 11/342: Type = BELONGS_TO, Source = assets, Target = non_controlling_shareholders
[INFO] Adding Relationship 12/342: Type = REPORTED_ON, Source = minority_interest, Target = consolidated_income_state

In [8]:
 Load the knowledge graph data from a JSON file
json_file_path = "/home/sbhavsar/PoisonedRAG/hybrid_approach/jsons/17_05_2025_knowledge_graph_new_sys.json"  # Path to your JSON file
with open(json_file_path, "r") as file:
    knowledge_graph = json.load(file)

# Store the knowledge graph in Neo4j
try:
    store_knowledge_graph(driver, knowledge_graph)
    print("Knowledge graph stored in Neo4j successfully!")
finally:
    driver.close()

SyntaxError: invalid syntax (3306092957.py, line 1)

In [ ]:
def get_all_node_labels():
    with driver.session() as session:
        result = session.run("CALL db.labels()")
        return [record["label"] for record in result]

In [ ]:
# Example usage
labels = get_all_node_labels()
print("Node Labels:", labels)

/tmp/ipykernel_63519/522634782.py:2: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  with driver.session() as session:


Node Labels: ['other', 'organization', 'document', 'person', 'financial_term', 'work', 'date', 'keyword', 'law', 'location', 'event', 'keyword_label', 'scientific_term', 'product', 'language', 'title', 'concept', 'character', 'award', 'project', 'facility', 'treatment', 'music', 'economic_term', 'economic_policy', 'policy', 'financial_instrument', 'activity']
